In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser

plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [2]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()
p_standard = pp / "mini_20x50x4__14_11/final"
p_sep = pp / 'separated_targets_20x50x4__26_11/final'
p_code = pp / 'mini_code_output_20x50__05_12/final'

In [3]:
mres_standard = MultiVariantMultiModelResultsAnalyser(p_standard)
mres_sep = MultiVariantMultiModelResultsAnalyser(p_sep)
mres_code = MultiVariantMultiModelResultsAnalyser(p_code)

Model: 100%|██████████| 20/20 [00:00<00:00, 122.99it/s]
/home/guests2/dkd/code/gsm-symbolic-benchmarking/src/gsm_benchmarker/results_analyser/multi_model.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(full_data_dict.values(), keys=full_data_dict.keys(), names=['model', 'old_index'])
Model: 100%|██████████| 20/20 [00:00<00:00, 57.88it/s]
/home/guests2/dkd/code/gsm-symbolic-benchmarking/src/gsm_benchmarker/results_analyser/multi_model.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat op

In [4]:
VARIANT = 'main'

## Comparison to baseline (GSM8K) questions

In [5]:
drop_standard = mres_standard.get_accuracy_drop_df(VARIANT)
drop_standard

accuracy_drop  strict_accuracy_drop
model                     id                                     
google_gemma-2-27b-it     0            0.00                 -0.05
                          1            0.00                  0.00
                          2            0.00                  0.35
                          3            0.00                  0.80
                          4           -0.05                 -0.05
...                                     ...                   ...
mistralai_Mistral-7B-v0_3 45          -0.25                  0.00
                          46          -0.25                  0.00
                          47          -0.05                  0.00
                          48           0.00                  0.00
                          49          -0.05                  0.00

[1000 rows x 2 columns]

In [6]:
drop_code = mres_code.get_accuracy_drop_df(VARIANT)

In [7]:
gap_closure_code = drop_standard - drop_code
gap_closure_code.rename(columns={'accuracy_drop': 'gap_closure', 'strict_accuracy_drop': 'strict_gap_closure'}, inplace=True)
gap_closure_code

gap_closure  strict_gap_closure
model                     id                                 
google_gemma-2-27b-it     0          0.00               -0.05
                          1          0.00                0.00
                          2          0.45                0.80
                          3         -0.15                0.65
                          4         -0.15               -0.15
...                                   ...                 ...
mistralai_Mistral-7B-v0_3 45        -0.25                0.00
                          46         0.00                0.00
                          47        -0.35                0.15
                          48         0.00                0.00
                          49         0.00                0.00

[1000 rows x 2 columns]

In [8]:
print(f"Mean Drop (Standard):\n{drop_standard.mean()}")
print(f"\nMean Drop (Code):\n{drop_code.mean()}")
print(f"\nMean Gap Closure:\n{gap_closure_code.mean()}")

Mean Drop (Standard):
accuracy_drop           0.06550
strict_accuracy_drop    0.03245
dtype: float64

Mean Drop (Code):
accuracy_drop           0.02960
strict_accuracy_drop    0.00865
dtype: float64

Mean Gap Closure:
gap_closure           0.0359
strict_gap_closure    0.0238
dtype: float64


In [9]:


def report_significance(control_drops, treatment_drops, p_threshold=0.05):
    t_stat, p_val_ttest = stats.ttest_rel(control_drops, treatment_drops, alternative='greater')
    print(f"P-value {p_val_ttest:.5f}")
    print(f"{'STATISTICALLY' if p_val_ttest < p_threshold else 'NOT'} SIGNIFICANT")

print(f"--- Paired T-Test ---")
print("\nLenient accuracy:")
report_significance(drop_standard.accuracy_drop, drop_code.accuracy_drop)

print("\nStrict accuracy:")
report_significance(drop_standard.strict_accuracy_drop, drop_code.strict_accuracy_drop)


--- Paired T-Test ---

Lenient accuracy:
P-value 0.00758
STATISTICALLY SIGNIFICANT

Strict accuracy:
P-value 0.01898
STATISTICALLY SIGNIFICANT


In [10]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

def analyze_drops_significance(df_control, df_treatment, col_name='drop'):
    """
    Analyzes whether the accuracy drop is significantly lower in the treatment group.

    Parameters:
    - df_control: DataFrame indexed by (model_name, template_id) containing accuracy drops.
    - df_treatment: DataFrame indexed by (model_name, template_id) containing accuracy drops.
    - col_name: The name of the column containing the drop values (e.g., 'acc_drop').
    """

    # Get list of unique models from the index level 0
    models = df_control.index.get_level_values(0).unique()

    results = []

    print(f"Processing {len(models)} models...\n")

    for model in models:
        # 1. Extract data for this specific model
        # We sort by index to ensure Template 1 aligns with Template 1
        try:
            ctrl_data = df_control.loc[model][col_name].sort_index()
            trt_data = df_treatment.loc[model][col_name].sort_index()
        except KeyError:
            print(f"Skipping {model}: Data missing in one of the dataframes.")
            continue

        # Check alignment
        if not ctrl_data.index.equals(trt_data.index):
            print(f"Warning: Template IDs do not match for {model}. Skipping.")
            continue

        # 2. Calculate the 'Gap Closure' (Difference of Differences)
        # We want to test if Control Drop > Treatment Drop
        # Therefore: Gap Closure = Control - Treatment > 0
        diffs = ctrl_data - trt_data

        # Handle edge case: If the method is identical to control (all diffs are 0)
        if np.all(diffs == 0):
            results.append({
                'Model': model,
                'Mean_Gap_Closure': 0.0,
                'Test_Used': 'None (Identical)',
                'P_Value': 1.0
            })
            continue

        # 3. Check Normality (Shapiro-Wilk)
        # If N < 50, Shapiro is best. If N > 5000, use K-S test.
        # Since N=50, Shapiro is appropriate.
        shapiro_stat, shapiro_p = stats.shapiro(diffs)
        is_normal = shapiro_p > 0.05

        # 4. Run the Statistical Test
        # We use 'greater' because we expect Control Drop > Treatment Drop
        if is_normal:
            test_name = 'Paired T-Test'
            stat, p_val = stats.ttest_rel(ctrl_data, trt_data, alternative='greater')
        else:
            test_name = 'Wilcoxon'
            # 'wilcoxon' excludes zero-differences by default (correction=True)
            stat, p_val = stats.wilcoxon(ctrl_data, trt_data, alternative='greater')

        results.append({
            'Model': model,
            'Mean_Gap_Closure': diffs.mean(),
            'Test_Used': test_name,
            'P_Value': p_val
        })

    # Convert results to DataFrame for easier manipulation
    results_df = pd.DataFrame(results)

    # 5. Apply Benjamini-Hochberg Correction
    # We only correct models where a test was actually run
    reject, pvals_corrected, _, _ = multipletests(results_df['P_Value'], alpha=0.05, method='fdr_bh')
    results_df['P_Value_Corrected'] = pvals_corrected
    results_df['Significant'] = reject

    return results_df


final_results = analyze_drops_significance(drop_standard, drop_code, col_name='accuracy_drop')

# 3. Display formatted output
final_results[['Model', 'Test_Used', 'Mean_Gap_Closure', 'P_Value', 'P_Value_Corrected', 'Significant']].round(4)

Processing 20 models...



,Model,Test_Used,Mean_Gap_Closure,P_Value,P_Value_Corrected,Significant
0,google_gemma-2-27b-it,Wilcoxon,0.009,0.3510,0.5850,False
1,google_gemma-2-2b,Wilcoxon,-0.013,0.4188,0.5983,False
2,google_gemma-2-2b-it,Wilcoxon,0.033,0.1153,0.3842,False
3,google_gemma-2-9b,Wilcoxon,0.186,0.0124,0.1159,False
4,google_gemma-2-9b-it,Wilcoxon,-0.005,0.4186,0.5983,False
5,google_gemma-2b,Wilcoxon,-0.050,0.7897,0.8774,False
6,google_gemma-2b-it,Wilcoxon,-0.012,0.2688,0.5376,False
7,google_gemma-7b,Wilcoxon,-0.102,0.8703,0.8874,False
8,google_gemma-7b-it,Wilcoxon,-0.014,0.6048,0.7560,False
9,meta-llama_Meta-Llama-3-8B,Paired T-Test,0.075,0.1639,0.4317,False
